# ch16 练习：时间序列处理

与讲解版逐 Cell 对应，`assert` 验收保留。卡住回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 数据准备及处理 | 重采样 + 日频特征 + 滞后特征 |
| 数据探索 | 字符串切片 + 节假日对比 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

load = pd.read_csv(DATA / "load_curve.csv")
load["时间戳"] = pd.to_datetime(load["时间戳"])

KEY = "台区编号"
VAL = "负荷值"
TS = "时间戳"

# 单台区小时级序列（DatetimeIndex）
st = load[load[KEY] == "STATION_A_01"].sort_values(TS).set_index(TS)

print("pandas", pd.__version__)
print("load", load.shape, "| 台区", load[KEY].nunique(),
      "| 时间范围", load[TS].min().date(), "~", load[TS].max().date())
print("st", st.shape, "| 负荷值 NaN", int(st[VAL].isna().sum()))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("st 索引类型:", type(st.index).__name__, "| 频率:", st.index.freq)

## 题 1：resample 频率与标签（对应讲解 §3.2）

In [ ]:
w = st.resample("W")[VAL].mean()
w_left = st.resample("W", label="left")[VAL].mean()

print("W 桶数:", len(w), "| 默认首桶:", w.index[0].date(),
      "| left 首桶:", w_left.index[0].date())
assert len(w) == 14
assert str(w.index[0].date()) == "2026-01-04"
assert str(w_left.index[0].date()) == "2025-12-28"

## 题 2：NaN 三分家（对应讲解 §3.3）

In [ ]:
#            负荷均值 NaN 桶数存入 n_nan
hr = st.resample("h")
n_buckets = len(hr.size())
n_empty = int((hr.size() == 0).sum())
n_nan = int(hr[VAL].mean().isna().sum())

daily = st.resample("D")[VAL].agg(["mean", "count", "max", "min"])
min_cnt = int(daily["count"].min())

print("小时桶:", n_buckets, "| 空桶:", n_empty, "| NaN 桶:", n_nan)
print("日均表:", daily.shape, "| 最小 count:", min_cnt)
assert n_buckets == 2160 and n_empty == 0 and n_nan == 23
assert daily.shape == (90, 4) and min_cnt == 18

## 题 3：pd.Grouper 分组重采样（对应讲解 §3.4）

In [ ]:
gd = load.groupby([KEY, pd.Grouper(key=TS, freq="D")])[VAL].mean()
wd = gd.unstack(0)

print("gd:", gd.shape, gd.index.names, "| wd:", wd.shape)
assert gd.shape == (270,) and gd.index.names == [KEY, TS]
assert wd.shape == (90, 3)
assert round(float(gd.iloc[0]), 4) == 484.035

## 题 4：日频特征四件套（对应讲解 §3.5）

In [ ]:
feat = st.copy()
feat["月份"] = feat.index.month
feat["小时"] = feat.index.hour
feat["星期"] = feat.index.dayofweek
feat["是否周末"] = (feat.index.dayofweek >= 5).astype(int)

print("月份:", sorted(feat["月份"].unique().tolist()),
      "| 周末行数:", int(feat["是否周末"].sum()))
assert sorted(feat["月份"].unique().tolist()) == [1, 2, 3]
assert int(feat["是否周末"].sum()) == 624
assert int(feat["星期"].value_counts()[2]) == 288

## 题 5：组内滞后（对应讲解 §3.6）

In [ ]:
#            diff(24) 生成「日环比」；再生成不分组版「不分组滞后」
g = load.sort_values([KEY, TS]).copy()
g["滞后1天"] = g.groupby(KEY)[VAL].shift(24)
g["日环比"] = g.groupby(KEY)[VAL].diff(24)
g["不分组滞后"] = g[VAL].shift(24)

print("滞后1天 NaN:", int(g["滞后1天"].isna().sum()),
      "| 日环比 NaN:", int(g["日环比"].isna().sum()))
print("错位行数:", int((g["滞后1天"].isna() != g["不分组滞后"].isna()).sum()))
print("验收 - 各组前 24 行全 NaN:",
      bool(g.groupby(KEY)["滞后1天"].head(24).isna().all()))
assert int(g["滞后1天"].isna().sum()) == 120
assert int(g["日环比"].isna().sum()) == 168
assert int((g["滞后1天"].isna() != g["不分组滞后"].isna()).sum()) == 48

## 题 6（综合）：字符串切片 + 节假日对比（对应讲解 §3.7-3.8）

In [ ]:
jan = st.loc["2026-01"]
day1 = st.loc["2026-01-01"]

hol_mean = load.loc[load["是否节假日"], VAL].mean()
work_mean = load.loc[~load["是否节假日"], VAL].mean()

print("1 月:", jan.shape, "| 单日:", day1.shape,
      "| 单日均值:", round(float(day1[VAL].mean()), 3))
print("节假日:", round(float(hol_mean), 3), "vs 工作日:", round(float(work_mean), 3))
assert jan.shape == (744, 4) and day1.shape == (24, 4)
assert round(float(day1[VAL].mean()), 3) == 484.035
assert round(float(hol_mean), 3) == 513.004
assert round(float(work_mean), 3) == 678.191

---

## 综合自查

1. 题 1 两种标签各适合什么报表场景？
2. 题 2 的 23 个 NaN 桶是缺口吗？用什么列证明？
3. 题 5 的 120 和 48 分别怎么算出来的？
4. 滞后特征的验收标准是什么？为什么重要？

全部答得上来，进入 ch17（性能与内存）。